# CEPHA29 — AI trace quality-gate calibration (Kaggle)

Produces the real distribution of the production model's **PSR confidence** and
**shape residual** on the held-out `valid` split, then prints suggested quality
thresholds for `src/lib/landmarkQuality.js`.

This does **not** train anything. It runs the offline harness
(`scripts/eval-landmark-model.py --report-ood`) on CPU against the shipped INT8
model.

**Kaggle setup**
1. Accelerator: **None** (CPU is fine — this is ONNX inference).
2. Internet: **On** (downloads the dataset + model).
3. Run All. At the end, download `/kaggle/working/cepha29-ood.csv`.

Optional: attach an existing Kaggle dataset containing `train|valid/Cephalograms`
to skip the ~1.95 GB download.

In [ ]:
# onnxruntime for inference; scipy for the DARK Gaussian blur.
!pip -q install onnxruntime scipy pillow numpy requests pandas
print("deps installed")

In [ ]:
import os
REPO_URL = "https://github.com/Muhammad-Nabeel-Sh/CephalometryStudio.git"
%cd /kaggle/working
!rm -rf CephalometryStudio
!git clone --depth 1 $REPO_URL CephalometryStudio
SCRIPT = "CephalometryStudio/scripts/eval-landmark-model.py"
assert os.path.exists(SCRIPT), "eval script missing from the clone"
# Confirm the clone actually has the --report-ood work (must be committed/pushed).
src = open(SCRIPT, encoding="utf-8").read()
assert "--report-ood" in src, "clone is missing --report-ood — push the latest commit first"
print("cloned; --report-ood present")

In [ ]:
import os, hashlib, zipfile, requests

def find_dataset(root):
    """Directory that contains a split dir (valid/test/train) with Cephalograms."""
    for base, dirs, _ in os.walk(root):
        for split in ("valid", "test", "train"):
            if os.path.isdir(os.path.join(base, split, "Cephalograms")):
                return base
    return None

DATASET = find_dataset("/kaggle/input")

# Else download Aariz (CEPHA29) from Figshare — CC BY 4.0.
if DATASET is None:
    URL = "https://ndownloader.figshare.com/files/51041642"
    MD5 = "e0bd645bca6759abdae4f199d841bda6"
    SIZE = 2098209792
    ZIP = "/kaggle/working/Aariz.zip"
    if not os.path.exists(ZIP) or os.path.getsize(ZIP) != SIZE:
        print("downloading Aariz.zip (~1.95 GB)…")
        with requests.get(URL, stream=True, headers={"User-Agent": "Mozilla/5.0"}) as r:
            r.raise_for_status()
            with open(ZIP, "wb") as f:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk)
    h = hashlib.md5()
    with open(ZIP, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == MD5, "Aariz.zip md5 mismatch — delete and re-run"
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/kaggle/working/aariz")
    DATASET = find_dataset("/kaggle/working/aariz")

print("DATASET =", DATASET)
assert DATASET, "Could not locate the dataset (need Internet On or an attached Kaggle dataset)."

In [ ]:
import os, glob

def pick_split(dataset):
    """First of valid/test/train that has BOTH images and annotation JSONs."""
    for split in ("valid", "test", "train"):
        imgs = os.path.join(dataset, split, "Cephalograms")
        if not os.path.isdir(imgs):
            continue
        ann_root = os.path.join(dataset, split, "Annotations")
        cands = []
        for base, _, _ in os.walk(ann_root):
            if glob.glob(os.path.join(base, "*.json")):
                cands.append(base)
        if not cands:
            continue
        # Prefer the Senior annotator (the shipped SSM prior was built from it).
        pref = [c for c in cands if "Senior" in c] or [c for c in cands if "Junior" in c] or cands
        return split, imgs, pref[0]
    return None

picked = pick_split(DATASET)
assert picked, "no split with images + annotations found"
SPLIT, IMAGES, ANN = picked
n_imgs = len([f for f in glob.glob(os.path.join(IMAGES, "*"))
              if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"))])
n_ann = len(glob.glob(os.path.join(ANN, "*.json")))
print(f"split      : {SPLIT}")
print(f"images     : {n_imgs}  ({IMAGES})")
print(f"annotations: {n_ann}  ({ANN})")
assert n_imgs and n_ann

In [ ]:
import os, hashlib, requests

MODEL_URL = "https://huggingface.co/MuhammadNabeelSh/cephalometry-landmarks/resolve/main/landmarks-cepha29.int8.onnx"
MODEL_SHA = "f7f1db9cc2de37c0fa94dab431d7d3446d1d8ffc62fa92e6610c2380af83080d"
MODEL = "/kaggle/working/landmarks-cepha29.int8.onnx"

if not os.path.exists(MODEL):
    print("downloading production INT8 model…")
    with requests.get(MODEL_URL, stream=True, headers={"User-Agent": "Mozilla/5.0"}) as r:
        r.raise_for_status()
        with open(MODEL, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)

h = hashlib.sha256()
with open(MODEL, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
print("model MB:", round(os.path.getsize(MODEL) / 1e6, 1))
print("sha256  :", h.hexdigest())
assert h.hexdigest() == MODEL_SHA, "model sha256 mismatch — delete and re-run"

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "CephalometryStudio/scripts/eval-landmark-model.py",
    "--model", MODEL,
    "--set", "cepha29",
    "--limit", "200",
    "--dark",                      # production decoding; PSR is measured on the blurred plane
    "--ann-dir", ANN,
    "--images", IMAGES,
    "--report-ood",
    "--ood-target", "0.05",        # aim for <=5% of good films rated "low"
    "--ood-out", "/kaggle/working/cepha29-ood.csv",
]
print(" ".join(cmd), "\n")
subprocess.run(cmd, cwd="/kaggle/working", check=True)

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/working/cepha29-ood.csv")
print(df.describe())
df.head(20)

## Next

1. Download `/kaggle/working/cepha29-ood.csv`.
2. Send the terminal report above **and** the CSV back — the suggested
   `DEFAULT_QUALITY_THRESHOLDS` block is printed at the end of the run.
3. Those numbers are then applied in `src/lib/landmarkQuality.js` (second commit)
   and the false-low rate is re-checked on this same `valid` split.

If the run reports a lot of `skip` lines, the image stems and annotation
filenames don't match — check the `images`/`annotations` counts printed above.